# Tau Supersymmetry — Analysis Guide

<div style="text-align: justify">

This notebook serves as a guide to the <b>Tau Supersymmetry</b> search analysis. The analysis is implemented as a modular, multi-stage machine learning pipeline — from raw ROOT data through multiclass classification to statistical significance testing. Each stage is available both as an <b>interactive Jupyter notebook</b> (with inline plots and outputs) and as a <b>CLI script</b> for automated, reproducible execution.

</div>

## Pipeline Overview

The diagram below shows how the stages connect. Each notebook can be explored independently — outputs from previous stages are persisted as Parquet files.

```
ROOT ntuples
     │
     ▼
┌─────────────────────┐
│  01  Preprocessing  │  ──▶  Selection cuts, event weights, ROOT → Parquet
└─────────┬───────────┘
          ▼
┌─────────────────────────┐
│  02  Feature Engineering│  ──▶  Physics features, class labels, sample merging
└─────────┬───────────────┘
          ▼
┌────────────────────────────┐
│  03  Exploratory Data      │  ──▶  Data quality, class balance, correlations,
│      Analysis              │       feature distributions
└─────────┬──────────────────┘
          ▼
┌─────────────────────────────┐
│  03b  Hyperparameter Tuning │  ──▶  Optuna search with k-fold CV and pruning
└─────────┬───────────────────┘
          ▼
┌─────────────────────────────────┐
│  04a  BDT Training (XGBoost)    │  ──▶  Gradient-boosted decision trees
│  04b  DNN Training (PyTorch)    │  ──▶  Deep neural network classifier
└─────────┬───────────────────────┘
          ▼
┌─────────────────────────────────┐
│  05a  BDT Evaluation            │  ──▶  Metrics, ROC, confusion matrix, SHAP
│  05b  DNN Evaluation            │  ──▶  Metrics, ROC, confusion matrix, SHAP
└─────────┬───────────────────────┘
          ▼
┌──────────────────────┐
│  06  ML-Based Regions│  ──▶  CR/VR/SR construction, CLs significance,
│                      │       kinematic distributions
└──────────────────────┘
```

## Notebooks

| # | Notebook | Description | CLI Equivalent |
|---|----------|-------------|----------------|
| 01 | [Preprocessing](./01_preprocessing.ipynb) | Reads ROOT ntuples, applies selection cuts, computes event weights, exports to Parquet | `python preprocess.py` |
| 02 | [Feature Engineering](./02_feature_engineering.ipynb) | Prepares physics-based input features, assigns class labels, merges samples | `python feature_engineer.py` |
| 03 | [Exploratory Data Analysis](./03_eda.ipynb) | Data quality checks, class balance, feature correlations and distributions | `python eda.py` |
| 03b | [Hyperparameter Tuning](./03b_hyperparameter_tuning.ipynb) | Optuna-based hyperparameter search with k-fold CV, TPE sampling, and pruning for BDT and DNN | `python tune.py` |
| 04a | [BDT Training (XGBoost)](./04a_bdt_training.ipynb) | Trains a gradient-boosted decision tree classifier with early stopping | `python train.py` |
| 04b | [DNN Training (PyTorch)](./04b_dnn_training.ipynb) | Trains a deep neural network with mini-batch SGD, AMP, and early stopping | `python train_dnn.py` |
| 05a | [BDT Evaluation](./05a_bdt_evaluation.ipynb) | Evaluation metrics, ROC curves, confusion matrix, SHAP feature importance | `python evaluate.py` |
| 05b | [DNN Evaluation](./05b_dnn_evaluation.ipynb) | Evaluation metrics, ROC curves, confusion matrix, SHAP feature importance | `python evaluate_dnn.py` |
| 06 | [ML-Based Regions](./06_ml_regions.ipynb) | Signal/control/validation region construction, CLs significance testing with pyhf, kinematic distributions | `python regions.py` |

## CLI Pipeline

The entire analysis can be run from the command line without opening any notebooks. Each script mirrors its notebook counterpart and is configured via [Hydra](https://hydra.cc/) YAML files in `configs/`.

```bash
# Run the full pipeline
make pipeline            # preprocess → feature-engineer → train

# Or run individual stages
make preprocess          # 01 — ROOT → Parquet
make feature-engineer    # 02 — feature extraction & sample merging
make eda                 # 03 — exploratory data analysis
make tune                # 03b — hyperparameter optimisation
make train               # 04a — BDT training
make regions             # 06 — ML-based regions & significance

# DVC-tracked reproducible run
make repro               # runs dvc repro
```

All hyperparameters can be overridden via the command line:

```bash
python train.py model.learning_rate=0.03 model.max_depth=5
python tune.py model=dnn tuning.n_trials=200 tuning.n_splits=3
```

## Experiment Tracking

All training and tuning runs are tracked with [MLflow](https://mlflow.org/). To browse experiments:

```bash
make ui    # starts MLflow UI at http://localhost:5000
```

## Stack

| Category | Libraries |
|----------|----------|
| ML & Training | XGBoost, PyTorch, scikit-learn, Optuna |
| Interpretability | SHAP |
| Statistical Analysis | pyhf (CLs hypothesis testing) |
| HEP Data I/O | uproot, awkward-array |
| Data Processing | pandas, NumPy, SciPy |
| Configuration | Hydra, OmegaConf |
| Experiment Tracking | MLflow |
| Pipeline & Versioning | DVC, Make |
| Visualisation | Matplotlib, Seaborn, Plotly |